# Planificacion automatica aplicada al Senku
## Experimentacion - Convocatoria de junio

Este cuaderno acompana al sistema desarrollado en `senku/src`. Sigue la metodologia de la Practica 4 de la asignatura: se utiliza la biblioteca `unified_planning` para parsear y representar problemas PDDL y `OneshotPlanner` con `Fast Downward` como referencia. Sobre esa misma infraestructura ejecutamos nuestras propias implementaciones de **BFS** (parte comun) y **Beam Search** con la **funcion pagoda** (algoritmo especifico de la convocatoria de junio).

Las variantes 1, 3 y 5 son las exigidas por el enunciado; las 2 y 4 se incluyen para experimentacion adicional.

In [ ]:
import sys
from pathlib import Path

# Permite ejecutar el notebook desde senku/notebooks/ sin instalar el paquete
RAIZ = Path.cwd().parent.parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from senku.src.tableros import TABLEROS, dibuja_tablero, VARIANTES_OBLIGATORIAS
from senku.src.estado import ProblemaSenku
from senku.src.heuristicas import (
    pagoda_clasica, pagoda_uniforme,
    heuristica_pagoda, heuristica_compuesta, valor_pagoda,
)
from senku.src.busqueda import (
    busqueda_primero_anchura,
    beam_search,
    beam_search_con_reinicios,
)
from senku.src.dominio_up import construye_problema_up
from senku.src.lector_pddl import carga_problema_pddl, carga_con_unified_planning
from senku.src.planificador import resuelve_con_fast_downward

print(f'Variantes definidas: {list(TABLEROS.keys())}')
print(f'Obligatorias (junio): {VARIANTES_OBLIGATORIAS}')

## 1. Inspeccion de los tableros

Visualizamos los cinco tableros con su estado inicial (`o` = casilla ocupada, `.` = hueco).

In [ ]:
for numero, tablero in TABLEROS.items():
    obligatoria = ' (obligatoria)' if numero in VARIANTES_OBLIGATORIAS else ''
    print(f'\n=== Variante {numero}{obligatoria}: {tablero.nombre} ({len(tablero.casillas)} casillas) ===')
    print(dibuja_tablero(tablero, tablero.inicial_ocupadas))

## 2. Pagoda de los estados iniciales

Comprobamos que la asignacion clasica de pagoda cumple la cota `a + b >= c` para todas las ternas de salto y calculamos la pagoda inicial y meta. Si no hubiera violaciones, podemos confiar en que la heuristica `h_pagoda` es admisible para esa variante.

In [ ]:
def valida_pagoda(pesos, problema):
    return [(d, s, h) for d, s, h in problema.saltos if pesos[d] + pesos[s] < pesos[h]]

for numero, tablero in TABLEROS.items():
    p = ProblemaSenku.desde_tablero(tablero)
    pesos = pagoda_clasica(tablero)
    pag_ini = valor_pagoda(p.inicial, pesos)
    pag_meta = sum(pesos[c] for c in p.meta_ocupadas if c in pesos)
    print(f'V{numero}: violaciones={len(valida_pagoda(pesos, p))} | '
          f'pagoda inicial={pag_ini} | pagoda meta={pag_meta} | exceso={pag_ini - pag_meta}')

## 3. Lectura del problema desde PDDL (estilo Practica 4)

Validamos el requisito de la convocatoria: el sistema recibe dos ficheros .pddl y los procesa con `unified_planning`. Los detalles del backend estan en `senku/src/lector_pddl.py`.

In [ ]:
ruta_dominio = RAIZ / 'senku' / 'pddl' / 'dominio_senku.pddl'
for numero in [1, 3, 5]:
    ruta_problema = RAIZ / 'senku' / 'pddl' / 'problemas' / f'variante_{numero}.pddl'
    p = carga_con_unified_planning(ruta_dominio, ruta_problema)
    print(f'V{numero}: {p.tablero.nombre} -> {len(p.tablero.casillas)} casillas, '
          f'{len(p.inicial)} piezas iniciales, {len(p.saltos)} saltos posibles')

## 4. Linea base: Fast Downward via unified-planning

Antes de evaluar nuestro Beam Search, comparamos contra Fast Downward (el planificador recomendado en la Practica 4). Esto nos permite saber, para cada variante, si la instancia es resoluble en absoluto y cuanto cuesta encontrar el plan optimo.

Si Fast Downward marca una variante como `UNSOLVABLE_INCOMPLETELY`, significa que ha demostrado por refutacion completa que no existe plan (por ejemplo, las variantes 2 y 4 son irresolubles por paridad).

In [ ]:
from concurrent.futures import ProcessPoolExecutor, TimeoutError as PFTimeout

def _resuelve_fd(variante):
    from unified_planning.io import PDDLReader
    from unified_planning.shortcuts import OneshotPlanner, get_environment
    get_environment().credits_stream = None
    import time
    raiz = Path.cwd().parent.parent
    p = PDDLReader().parse_problem(
        str(raiz/'senku/pddl/dominio_senku.pddl'),
        str(raiz/f'senku/pddl/problemas/variante_{variante}.pddl'))
    inicio = time.perf_counter()
    r = OneshotPlanner(name='fast-downward').solve(p)
    return {
        'variante': variante,
        'estado': str(r.status).split('.')[-1],
        'movimientos': len(r.plan.actions) if r.plan else 0,
        'tiempo_s': round(time.perf_counter()-inicio, 2),
    }

# Cada variante en un proceso con timeout para evitar bloqueos.
TIMEOUT = 90
filas_fd = []
for v in [1, 2, 3, 4, 5]:
    with ProcessPoolExecutor(max_workers=1) as exe:
        futuro = exe.submit(_resuelve_fd, v)
        try:
            r = futuro.result(timeout=TIMEOUT)
        except PFTimeout:
            r = {'variante': v, 'estado': 'TIMEOUT', 'movimientos': 0, 'tiempo_s': TIMEOUT}
            for proc in exe._processes.values():
                proc.terminate()
    print(r)
    filas_fd.append(r)

## 5. BFS y Beam Search propios sobre los mismos PDDL

Cargamos cada problema PDDL con `unified_planning`, lo convertimos a nuestra representacion interna y aplicamos los dos algoritmos implementados a mano.

In [ ]:
import pandas as pd

filas = []
LIMITE_NODOS_BFS = 50_000
BETA = 200
INTENTOS = 5
ITER_MAX = 80

for numero, tablero in TABLEROS.items():
    ruta_problema = RAIZ / 'senku' / 'pddl' / 'problemas' / f'variante_{numero}.pddl'
    p = carga_problema_pddl(ruta_dominio, ruta_problema)
    pesos = pagoda_clasica(tablero)
    h_pag = heuristica_pagoda(p, pesos)
    h_com = heuristica_compuesta(p, pesos)

    r = busqueda_primero_anchura(p, limite_nodos=LIMITE_NODOS_BFS)
    filas.append({'variante': numero, 'algoritmo': 'BFS', 'heuristica': '-',
                  'exito': r.exito, 'movimientos': len(r.movimientos),
                  'nodos': r.nodos_expandidos, 'tiempo_s': round(r.tiempo_segundos, 3)})

    r = beam_search_con_reinicios(p, h_pag, beta=BETA, intentos=INTENTOS, iteraciones_maximas=ITER_MAX)
    filas.append({'variante': numero, 'algoritmo': 'Beam', 'heuristica': 'pagoda',
                  'exito': r.exito, 'movimientos': len(r.movimientos),
                  'nodos': r.nodos_expandidos, 'tiempo_s': round(r.tiempo_segundos, 3)})

    r = beam_search_con_reinicios(p, h_com, beta=BETA, intentos=INTENTOS, iteraciones_maximas=ITER_MAX)
    filas.append({'variante': numero, 'algoritmo': 'Beam', 'heuristica': 'compuesta',
                  'exito': r.exito, 'movimientos': len(r.movimientos),
                  'nodos': r.nodos_expandidos, 'tiempo_s': round(r.tiempo_segundos, 3)})

df = pd.DataFrame(filas)
df

## 6. Influencia del parametro beta

Beam Search es incompleto y su capacidad de encontrar solucion depende mucho de la anchura del haz. Este experimento mide el efecto de beta sobre la profundidad alcanzada y el numero de exitos en multiples reinicios.

In [ ]:
VARIANTE_OBJETIVO = 1  # cruz inglesa
BETAS = [50, 100, 200, 500, 1000, 2000]
INTENTOS_BARRIDO = 5

p = ProblemaSenku.desde_tablero(TABLEROS[VARIANTE_OBJETIVO])
pesos = pagoda_clasica(p.tablero)
h = heuristica_compuesta(p, pesos)

filas_beta = []
for beta in BETAS:
    r = beam_search_con_reinicios(p, h, beta=beta, intentos=INTENTOS_BARRIDO, iteraciones_maximas=60)
    filas_beta.append({'beta': beta, 'exito': r.exito, 'mov': len(r.movimientos),
                       'nodos': r.nodos_expandidos, 'tiempo_s': round(r.tiempo_segundos, 3)})
pd.DataFrame(filas_beta)

## 7. Modo relajado

El enunciado menciona versiones relajadas en las que basta con terminar con una pieza en cualquier sitio. Re-ejecutamos beam search bajo este criterio.

In [ ]:
filas_relax = []
for numero, tablero in TABLEROS.items():
    p = ProblemaSenku.desde_tablero(tablero, modo_relajado=True)
    pesos = pagoda_clasica(tablero)
    h = heuristica_compuesta(p, pesos)
    r = beam_search_con_reinicios(p, h, beta=500, intentos=5, iteraciones_maximas=60)
    filas_relax.append({'variante': numero, 'casillas': len(tablero.casillas),
                        'exito': r.exito, 'mov': len(r.movimientos),
                        'nodos': r.nodos_expandidos, 'tiempo_s': round(r.tiempo_segundos, 3)})
pd.DataFrame(filas_relax)

## 8. Conclusiones experimentales

1. **BFS no escala**: el espacio de estados crece exponencialmente y BFS no encuentra solucion en las variantes medianas con un presupuesto razonable de nodos.
2. **Fast Downward sirve como referencia**: confirma irresolubilidad estructural en variantes con parida adversa (V2 y V4) y resuelve V1 / V3 / V5 (cuando tiene presupuesto suficiente).
3. **Beam Search con pagoda es incompleto**: aunque la heuristica es admisible, no contiene suficiente senal discriminativa para guiar la busqueda hasta el plan.
4. **La heuristica compuesta y los reinicios estocasticos mitigan parcialmente** la incompletud: al penalizar piezas aisladas el algoritmo escapa de callejones sin salida y alcanza mayor profundidad media.